# MLS French preprocessing

Builds a HuggingFace `datasets.Dataset` from one split of MLS French + the MFA TextGrid alignments produced upstream in the Snakefile. Mirrors the LibriSpeech preprocessing notebook but skips syllabification (no French syllabifier) and matches the English pipeline's explicit `Wav2Vec2FeatureExtractor` settings.

In [ ]:
split = "train"
data_dir = f"data/mls_french/{split}"
alignment_dir = f"data/mfa_alignments/mls_french/{split}"
out_path = "tst"
drop_phones = ["sil", "sp", "spn", ""]

In [ ]:
import logging
import unicodedata
from copy import copy

import datasets
import numpy as np
import transformers

In [ ]:
L = logging.getLogger(__name__)
L.setLevel(logging.ERROR)
datasets.disable_caching()

In [ ]:
dataset = datasets.load_dataset(
    "src/datasets/huggingface_mls_french.py",
    data_dir=data_dir,
    alignment_dir=alignment_dir,
    trust_remote_code=True)
# The builder yields a single split per invocation; pull it out of the DatasetDict.
dataset = next(iter(dataset.values()))
print(dataset)

In [ ]:
# DEV
dataset = dataset.select(range(100))

In [ ]:
# Explicit construction to match the English (LibriSpeech) pipeline byte-for-byte:
# input-value distributions need to be comparable across languages if we ever stack
# analogy results across them, and the FR checkpoint's shipped preprocessor_config
# may differ in do_normalize / return_attention_mask. No tokenizer needed because
# downstream code only consumes input_values + alignments.
feature_extractor = transformers.Wav2Vec2FeatureExtractor(
    feature_size=1, sampling_rate=16000, padding_value=0.0,
    do_normalize=True, return_attention_mask=False)
print(feature_extractor)

In [ ]:
def add_phonemic_detail(item):
    # MFA phones are already phonemic (no stress markers); copy verbatim for
    # consistency with the LibriSpeech schema that downstream code reads.
    item["phonemic_detail"] = {
        "start": copy(item["phonetic_detail"]["start"]),
        "stop":  copy(item["phonetic_detail"]["stop"]),
        "utterance": [unicodedata.normalize("NFC", u)
                      for u in item["phonetic_detail"]["utterance"]],
    }
    return item

In [ ]:
def group_phonetic_detail(item, idx, drop_phones=None, key="phonetic_detail"):
    """Group phones by their containing word, mirroring the LibriSpeech preprocessor."""
    phonetic_detail = item[key]
    word_detail = item["word_detail"]

    phone_mask = np.zeros(len(phonetic_detail["start"]), dtype=bool)
    word_phonetic_detail = []
    for start, stop, word in zip(word_detail["start"], word_detail["stop"], word_detail["utterance"]):
        word_phonetic_detail.append([])
        for j, (phon_start, phon_stop, phon) in enumerate(
                zip(phonetic_detail["start"], phonetic_detail["stop"], phonetic_detail["utterance"])):
            if phone_mask[j]:
                continue
            if drop_phones is not None and phon in drop_phones:
                phone_mask[j] = True
                continue
            if phon_start >= start and phon_start < stop:
                phone_mask[j] = True
                word_phonetic_detail[-1].append(
                    {"phone": phon, "start": phon_start, "stop": phon_stop})

        if not word_phonetic_detail[-1] and word != "":
            L.warning(f"No phones grouped under word {word!r} in item {idx} ({item['text']!r})")

    unused = np.flatnonzero(~phone_mask)
    if len(unused) > 0:
        L.warning(f"{len(unused)} phones ungrouped in item {idx} ({item['text']!r})")

    item[f"word_{key}"] = word_phonetic_detail
    return item

In [ ]:
def normalize_strings(item):
    # Defensive NFC + lowercase at the dataset level. The builder already NFCs,
    # but this guards against any downstream concatenation of un-normalized data.
    item["text"] = unicodedata.normalize("NFC", item["text"]).lower()
    item["word_detail"]["utterance"] = [
        unicodedata.normalize("NFC", u).lower()
        for u in item["word_detail"]["utterance"]]
    item["phonetic_detail"]["utterance"] = [
        unicodedata.normalize("NFC", u)
        for u in item["phonetic_detail"]["utterance"]]
    return item

In [ ]:
def prepare_audio(batch):
    audio = batch["audio"]
    batch["input_values"] = feature_extractor(
        audio["array"], sampling_rate=audio["sampling_rate"]).input_values[0]
    return batch

In [ ]:
def add_idx(item, idx):
    item["idx"] = idx
    item["split"] = split
    return item

In [ ]:
dataset = dataset.map(normalize_strings, num_proc=8)

In [ ]:
dataset = dataset.map(add_phonemic_detail, num_proc=8)
dataset = dataset.map(group_phonetic_detail, with_indices=True,
                      fn_kwargs=dict(drop_phones=drop_phones, key="phonetic_detail"), num_proc=8)
dataset = dataset.map(group_phonetic_detail, with_indices=True,
                      fn_kwargs=dict(drop_phones=drop_phones, key="phonemic_detail"), num_proc=8)

In [ ]:
dataset = dataset.map(prepare_audio, num_proc=8)
dataset = dataset.map(add_idx, with_indices=True, num_proc=8)

In [ ]:
dataset.save_to_disk(out_path)